# Causal Analysis

In [1]:
from CodeSmells import pos_utils as pos_utils

In [2]:
def default_params(): 
    return {
        'input_path': '/workspaces/mica/semeru-datasets/mica/causal_analysis',
        'causal_analysis': {
            'potential_outcome': 'code_smell_psc_relative', #code_smell_actual_prob_median, #code_smell_max_prob_median, #code_smell_min_prob_median, #code_smell_actual_prob_mean, #code_smell_max_prob_mean, #code_smell_min_prob_mean, #code_smell_psc_entropy, #code_smell_psc_relative  
            'common_causes': lambda causes: causes,
            'effect_modifiers' : lambda effect_modifiers: effect_modifiers,
            'instruments' : lambda instruments: instruments
        }, 
        ######## FOR CAUSAL ANALYSIS ########
        'features': {
            'syntactic' : ['complexity', 'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers'],
            'semantic' : [] + pos_utils.UNIVERSAL_SEMANTIC_TAGS,
        },
        'treatment' : 'T1', # [T1 (generation_type), T2 (model_size), T3 (model_architecture), T4 (prompt)]
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
    }
params = default_params()

### Imports

In [3]:
import pandas as pd
import numpy as np
from dowhy import CausalModel
import dowhy.datasets
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import zscore
import json
import os
import scipy.stats as stats

In [4]:
import matplotlib.pyplot as plt

### Dataset loading

In [18]:
combined_control_df = pd.read_json(F"{params['input_path']}/{params['model']}/control.json")
treatment_control_df = pd.read_json(F"{params['input_path']}/{params['model']}/treatment.json")

In [19]:
combined_control_df['group'] = 'control'
treatment_control_df['group'] = 'treatment'

In [20]:
causal_hypothesis_df =  pd.concat([combined_control_df, treatment_control_df], ignore_index=True)

In [21]:
causal_hypothesis_df.columns

Index(['method_id', 'java_code', 't_length_token', 'java_code_infered', 'loss',
       'min_k', 'zlib', 'relative_loss', 'relative_zlib', 'relative_min_k',
       't_complexity', 't_nloc', 't_token_counts', 't_ast_levels',
       't_n_ast_nodes', 't_n_ast_errors', 't_ast_errors', 't_n_identifiers',
       't_identifiers', 'e_complexity', 'e_nloc', 'e_token_counts',
       'e_ast_levels', 'e_n_ast_nodes', 'e_n_ast_errors', 'e_ast_errors',
       'e_n_identifiers', 'e_identifiers', 'group', 'model', 'rule_number'],
      dtype='object')

### Causal Modeling

In [22]:
class RefutationResult:
    def __init__(self, refutation_result=None, new_effect=None):
        self.refutation_result = refutation_result
        self.new_effect = new_effect

In [23]:
def compute_ATE_and_refute(causal_model, identified_estimand, method_name, method_params={}):
    try:
        estimate = causal_model.estimate_effect(identified_estimand, method_name, method_params = method_params, test_significance=True)
    except Exception as e:
        print(f"======= ERROR COMPUTING CE for {method_name}: {e} ============")
        return {'method_name': method_name, 
            'estimated_effect': None,
            'refutation_placebo_permute' : None, 
            'refutation_unobserved_confounder' : None, 
            'refutation_subset' : None}
    
    refutation_placebo_permute = RefutationResult()
    refutation_unobserved_confounder = RefutationResult()
    refutation_subset = RefutationResult()
    refutation_random_common_cause = RefutationResult()
    
    try:
        refutation_placebo_permute = causal_model.refute_estimate(
                identified_estimand, estimate, method_name="placebo_treatment_refuter", placebo_type="permute")
    except Exception as e:
        print(f"Error performing pacebo refutation - {e}")
    try: 
        refutation_unobserved_confounder = causal_model.refute_estimate(
                identified_estimand, estimate, method_name="add_unobserved_common_cause")
    except Exception as e:
        print(f"Error performing unobserved covariate refutation - {e}")
    try:
        refutation_subset = causal_model.refute_estimate(
                identified_estimand, estimate, method_name="data_subset_refuter")
    except Exception as e:
        print(f"Error performing subset refutation - {e}")
    try: 
        refutation_random_common_cause = causal_model.refute_estimate(
                identified_estimand, estimate, method_name="random_common_cause")
    except Exception as e:
        print(f"Error performing random covariate refutation - {e}")
    

    return {'method_name': method_name, 
            'estimated_effect': estimate.value,
            'refutation_random_common_cause' : {'result' : refutation_random_common_cause.refutation_result, 'new_effect': refutation_random_common_cause.new_effect},
            'refutation_placebo_permute' : {'result' : refutation_placebo_permute.refutation_result, 'new_effect': refutation_placebo_permute.new_effect}, 
            'refutation_unobserved_confounder' : {'result' : refutation_unobserved_confounder.refutation_result, 'new_effect': refutation_unobserved_confounder.new_effect}, 
            'refutation_subset' : {'result' : refutation_subset.refutation_result, 'new_effect': refutation_subset.new_effect},}

def compute_correlations(input_corr_data, outout_corr_data):
    pearson_corr =  stats.pearsonr(input_corr_data, outout_corr_data)
    spearman_corr = stats.spearmanr(input_corr_data, outout_corr_data)
    kendall_corr = stats.kendalltau(input_corr_data, outout_corr_data)
    return {'pearson_corr' : pearson_corr.statistic, 'spearman_corr' : spearman_corr.statistic, 'kendall_corr' : kendall_corr.statistic}

def compute_causal_effects(causal_model):
    causal_effects_df = pd.DataFrame(columns=['method_name', 'pearson_corr', 'spearman_corr', 'kendall_corr' ,'estimated_effect', 'refutation_placebo_permute', 'refutation_unobserved_confounder', 'refutation_subset'])
    ####### COMPUTE PEARSON
    #correlation_results = compute_correlations(list(causal_model._data[causal_model._common_causes].mean(axis=1)), list(causal_model._data[causal_model._outcome].mean(axis=1)))
    correlation_results = compute_correlations(causal_model._data['binary_treatment'].tolist(), list(causal_model._data[causal_model._outcome].mean(axis=1)))
    ####### COMPUTE ESTIMAND
    identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)
    ###### COMPUTE CAUSAL EFFECT - BACKDOOR - propensity_score_matching
    causal_effects_df.loc[len(causal_effects_df)] = {**correlation_results, **compute_ATE_and_refute(causal_model, identified_estimand, "backdoor.propensity_score_matching")}
    ###### COMPUTE CAUSAL EFFECT - BACKDOOR - propensity_score_stratification
    causal_effects_df.loc[len(causal_effects_df)] = {**correlation_results, **compute_ATE_and_refute(causal_model, identified_estimand, "backdoor.propensity_score_stratification")}
    ###### COMPUTE CAUSAL EFFECT - BACKDOOR - backdoor.propensity_score_weighting
    causal_effects_df.loc[len(causal_effects_df)] = {**correlation_results, **compute_ATE_and_refute(causal_model, identified_estimand, "backdoor.propensity_score_weighting")}
    ###### COMPUTE CAUSAL EFFECT - BACKDOOR - backdoor.propensity_score_weighting
    causal_effects_df.loc[len(causal_effects_df)] = {**correlation_results, **compute_ATE_and_refute(causal_model, identified_estimand, "backdoor.generalized_linear_model", method_params={"glm_family": sm.families.Gaussian()})}
    
    return causal_effects_df.dropna()


In [24]:
def compute_mia_causal_effects(causal_hypothesis_df, outcomes):
    causal_effect_dfs = []
    for rule_number in list(causal_hypothesis_df['rule_number'].unique()):
        print(f"=========== CAUSAL ANALYSIS FOR RULE-{rule_number}")
        #### EXTRACT RULE
        rule_df = causal_hypothesis_df[causal_hypothesis_df['rule_number']==rule_number].copy()
        rule_df['binary_treatment'] = rule_df['group'].map(lambda group: True if group !='control' else False)
        rule_df = rule_df.dropna()
        ### ZCORE FOR OUTCOMES
        #for outcome in outcomes: rule_df[outcome] = zscore(rule_df[outcome])
        ##### DEFINE CAUSAL MODEL
        causal_model = CausalModel(
            data = rule_df,
            treatment = ['binary_treatment'],
            outcome = outcomes,
            common_causes = params['causal_analysis']['common_causes'],
            effect_modifiers = params['causal_analysis']['effect_modifiers'],
            instruments = params['causal_analysis']['instruments'])
        ##### STORE RESULTS
        result_df = compute_causal_effects(causal_model)
        result_df['rule_number'] = rule_number
        causal_effect_dfs.append(result_df)
    return pd.concat(causal_effect_dfs, ignore_index=True)



In [25]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [26]:
len(list(causal_hypothesis_df['rule_number'].unique()))

### EXECUTE

In [27]:
def execute_analysis(outcomes, causal_hypothesis_df):
    for outcome in outcomes:
        print(f"============================= COMPUTING ANALYSIS FOR OUTCOME-{outcome} ==================================== ")
        causal_effects_outcome_df = compute_mia_causal_effects(causal_hypothesis_df, [outcome])
        print(f"============================= STORING ANALYSIS FOR OUTCOME-{outcome} ==================================== ")
        output_path = f"{params['output_path']}/{params['model']}"
        create_folder(output_path)
        causal_effects_outcome_df.to_json(f"{output_path}/{[outcome]}.json")
    #print(f"============================= COMPUTING ANALYSIS FOR ALL OUTCOMES-{outcomes} ==================================== ")
    #causal_effects_outcome_df = compute_mia_causal_effects(causal_hypothesis_df, outcomes)
    #print(f"============================= STORING ANALYSIS FOR ALL OUTCOMES-{outcomes}==================================== ")
    #output_path = f"{params['output_path']}/{params['model']}"
    #create_folder(output_path)
    #causal_effects_outcome_df.to_json(f"{output_path}/{outcomes}.json")

In [28]:
list(causal_hypothesis_df['rule_number'].unique())

['rule_4',
 'rule_22',
 'rule_10',
 'rule_17',
 'rule_9',
 'rule_3',
 'rule_7',
 'rule_21',
 'rule_13',
 'rule_19',
 'rule_14',
 'rule_16',
 'rule_2',
 'rule_8',
 'rule_5',
 'rule_23',
 'rule_11',
 'rule_15',
 'rule_all',
 'rule_1',
 'rule_6',
 'rule_20',
 'rule_18',
 'rule_12']

In [ ]:
execute_analysis(['loss', 'min_k', 'zlib', 'relative_loss', 'relative_min_k', 'relative_zlib'], causal_hypothesis_df)

============================= COMPUTING ANALYSIS FOR OUTCOME-loss ==================================== 


=========== CAUSAL ANALYSIS FOR RULE-rule_4


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
